In [30]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
import os
llm = ChatGroq(
    temperature=0,
    groq_api_key=os.getenv("API_KEY"),
    model_name="meta-llama/llama-4-scout-17b-16e-instruct"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The best answer is Neil Armstrong


In [33]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.teamviewer.com/jobs/7320032-staff-software-engineer-full-stack")
page_data = loader.load().pop().page_content
print(page_data)





















      Staff Software Engineer - Full Stack - TeamViewer Germany GmbH
    





























This website uses cookies to ensure you get the best experience.

        TeamViewer Germany GmbH and our selected partners use cookies and similar technologies (together “cookies”) that are necessary to present this website, and to ensure you get the best experience of it.

          If you consent to it, we will also use cookies for analytics purposes.
      
See our Cookie Policy to read more about the cookies we set.
You can withdraw and manage your consent at any time, by clicking “Manage cookies” at the bottom of each website page.


Accept all cookies
Decline all non-necessary cookies
Cookie preferences




Skip to main content




















Career menu








        Employee
      
Log in as employee

          Candidate
        
Log in to Connect
Homepage
teamviewer.com
 









Research & Development
·
Noida
·

    Hybrid
    


Staff Software E

In [35]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

```json
{
  "role": "Staff Software Engineer - Full Stack",
  "experience": "9+ years of professional experience in the field of software development",
  "skills": [
    "Typescript",
    "JavaScript",
    "HTML",
    "CSS",
    "React",
    "Redux",
    "Webpack",
    "Kubernetes",
    "Docker",
    "Kafka",
    "Argo CD",
    "Azure/AWS/GCP services",
    "Java",
    "C#",
    "C++",
    "Cypress UI test framework",
    "Unit test frameworks"
  ],
  "description": "We’re looking for a Staff Software Engineer to play a key role in supporting and enhancing our TeamViewer ONE capabilities, trusted by small and medium business customers. In this role, you’ll be essential to maintaining and improving the experience for our existing customers, while helping evolve our User Experience and SaaS cloud platform. You will also bring in expertise in using technologies - like React, Typescript, Javascript, CSS, Open API, Redux, Webpack. As part of an agile team, you’ll contribute as an individual

In [36]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Staff Software Engineer - Full Stack',
 'experience': '9+ years of professional experience in the field of software development',
 'skills': ['Typescript',
  'JavaScript',
  'HTML',
  'CSS',
  'React',
  'Redux',
  'Webpack',
  'Kubernetes',
  'Docker',
  'Kafka',
  'Argo CD',
  'Azure/AWS/GCP services',
  'Java',
  'C#',
  'C++',
  'Cypress UI test framework',
  'Unit test frameworks'],
 'description': 'We’re looking for a Staff Software Engineer to play a key role in supporting and enhancing our TeamViewer ONE capabilities, trusted by small and medium business customers. In this role, you’ll be essential to maintaining and improving the experience for our existing customers, while helping evolve our User Experience and SaaS cloud platform. You will also bring in expertise in using technologies - like React, Typescript, Javascript, CSS, Open API, Redux, Webpack. As part of an agile team, you’ll contribute as an individual contributor, working primarily with Typescript, react

In [37]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [38]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [39]:
links = collection.query(query_texts=json_res['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/typescript-frontend-portfolio'},
  {'links': 'https://example.com/full-stack-js-portfolio'}],
 [{'links': 'https://example.com/full-stack-js-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/wordpress-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/wordpress-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/react-native-portfolio'}],
 [{'links': 'https://example.com/full-stack-js-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/full-stack-js-portfolio'},
  {'links': 'https://example.com/react-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/kotlin-backend-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'

In [40]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Akash, currently working as an intern at Airtel with experience in AI, software development, and building automation solutions.

Your task is to write a personalized cold email to a potential client regarding the job mentioned above. In the email, explain how your technical skills, problem-solving ability, and experience working with modern technologies can help address the client's requirements.

Highlight your interest in collaborating, building scalable solutions, and delivering efficient results. Maintain a professional, concise, and persuasive tone.

Also include the most relevant portfolio links from the following list to showcase your work and projects:
{link_list}

Guidelines:
- The email should be professional and concise.
- Clearly connect the client's requirements with your skills and experience.
- Briefly introduce yourself and your background.
- Naturally include the relevant portfolio links.

Remember you are Akash, an intern at Airtel.
Do not include any preamble or explanation.
Only output the final cold email.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(json_res), "link_list": links})
print(res.content)

Subject: Application for Staff Software Engineer - Full Stack Position

Dear Hiring Manager,

I am Akash, an intern at Airtel with a strong background in AI, software development, and building automation solutions. I came across the Staff Software Engineer - Full Stack position at TeamViewer and was impressed by the opportunity to contribute to the enhancement of TeamViewer ONE capabilities.

With my experience in software development and passion for modern technologies, I am confident that I can bring significant value to your team. My technical skills align well with the job requirements, including proficiency in JavaScript, React, and TypeScript. I have hands-on experience with full-stack development, and my projects showcase my ability to build scalable and efficient solutions.

I am particularly drawn to this role because of the opportunity to work with cutting-edge technologies such as Kubernetes, Docker, and Redux. My experience with containerization and agile development method